# Quantile Forecasting (padronizado)


*By Miguel Ferreira*

**Este notebook, assim com todos os outros de cada ferramenta do envelope de risco, segue o mesmo protocolo:**
1. Importação do dataset e bibliotecas
2. Execução do ```setup()``` e alinhamento temporal
3. Construção da ferramenta de risco
4. Sanity checks mínimos
5. DataFrame final (features de risco) 
6. Padronização
7. Salvamento

Esta padronização é uma peça fundamental para um projeto **clean code.** Tanto que esta introdução estará presente em todos os notebooks de todas as ferramentas do envelope de risco.

---
A ferramenta deste notebook é o Quantile Forecasting.
> O Quantile Forecasting é uma ferramenta de risco **preditivo**

Isto a torna um pouco diferente de todas as outras, que são de *output pontual* (entregam uma análise para agora e não para o futuro).


## 1) Importação do dataset e bibliotecas


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path("../../src").resolve()))
from setup import setup

CSV_PATH = "C:/projects/Libellula/data/processed/financial_tools_datset.csv"
TARGET_COL = "Price"
HORIZON = 1
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15


## 2) Execução do `setup()` e alinhamento temporal


In [2]:
raw_df = pd.read_csv(CSV_PATH)
raw_df["Date"] = pd.to_datetime(raw_df["Date"], format="%m/%d/%Y")
raw_df = raw_df.sort_values("Date").set_index("Date")

splits = setup(
    csv_path=CSV_PATH,
    target_col=TARGET_COL,
    horizon=HORIZON,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    save_artifacts=False,
)

processed_df = raw_df.reset_index().copy()
if "Change %" in processed_df.columns:
    processed_df["Change %"] = (
        processed_df["Change %"].str.replace("%", "", regex=False).astype(float) / 100
    )

processed_df = processed_df.sort_values("Date")
processed_df["target_return"] = (
    processed_df[TARGET_COL].pct_change(HORIZON).shift(-HORIZON)
)
processed_df = processed_df.dropna().set_index("Date")

n_rows = len(processed_df)
train_end = int(n_rows * TRAIN_RATIO)
val_end = int(n_rows * (TRAIN_RATIO + VAL_RATIO))
val_index = processed_df.iloc[train_end:val_end].index

print("Validation length:", len(val_index))


Validation length: 205


## 3) Construção da ferramenta de risco (quantile forecasting)


In [3]:
from sklearn.ensemble import GradientBoostingRegressor

X_train = splits["X_train"]
X_val = splits["X_val"]
y_train = splits["y_train"]

quantiles = [0.05, 0.25, 0.50, 0.75, 0.95]
preds = {}

for q in quantiles:
    model = GradientBoostingRegressor(
        loss="quantile",
        alpha=q,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    )
    model.fit(X_train, y_train)
    preds[q] = model.predict(X_val)

qf_dataset = pd.DataFrame(index=val_index)
qf_dataset["quantile_lower"] = preds[0.05]
qf_dataset["q25"] = preds[0.25]
qf_dataset["q50"] = preds[0.50]
qf_dataset["q75"] = preds[0.75]
qf_dataset["quantile_upper"] = preds[0.95]

qf_dataset["quantile_width"] = qf_dataset["quantile_upper"] - qf_dataset["quantile_lower"]
qf_dataset["quantile_skew"] = (
    (qf_dataset["q75"] - qf_dataset["q50"]) - (qf_dataset["q50"] - qf_dataset["q25"])
)

safe_width = qf_dataset["quantile_width"].replace(0, np.nan)
qf_dataset["asymmetry"] = (qf_dataset["q50"] - qf_dataset["quantile_lower"]) / safe_width
qf_dataset["asymmetry"] = qf_dataset["asymmetry"].replace([np.inf, -np.inf], np.nan).fillna(0.5)

qf_dataset = qf_dataset.sort_index()


## 4) Sanity checks mínimos


In [4]:
print("NaN críticos:")
print(qf_dataset[["quantile_lower", "q50", "quantile_upper", "quantile_width"]].isna().sum())

print("\nDistribuição plausível:")
print(qf_dataset.describe().T[["mean", "std", "min", "max"]])

print("\nAlinhamento temporal:")
print("Index monotonic increasing:", qf_dataset.index.is_monotonic_increasing)
print("Index has duplicates:", qf_dataset.index.has_duplicates)

print("Dataset lines == val_index lines:", len(qf_dataset) == len(val_index))
print("Dataset index == val_index:", qf_dataset.index.equals(val_index))


NaN críticos:
quantile_lower    0
q50               0
quantile_upper    0
quantile_width    0
dtype: int64

Distribuição plausível:
                    mean       std       min       max
quantile_lower -0.007089  0.001283 -0.009241 -0.001417
q25            -0.002833  0.001090 -0.006914  0.000058
q50            -0.000056  0.001143 -0.004779  0.003528
q75             0.002486  0.000900  0.000259  0.005138
quantile_upper  0.007086  0.000959  0.006149  0.008689
quantile_width  0.014174  0.002000  0.007566  0.017623
quantile_skew  -0.000235  0.001482 -0.006062  0.005762
asymmetry       0.484860  0.140008 -0.155036  0.795637

Alinhamento temporal:
Index monotonic increasing: True
Index has duplicates: False
Dataset lines == val_index lines: True
Dataset index == val_index: True


## 5) DataFrame final (features de risco) 


In [5]:
qf_dataset = qf_dataset[[
    "quantile_lower",
    "q25",
    "q50",
    "q75",
    "quantile_upper",
    "quantile_width",
    "quantile_skew",
    "asymmetry",
]]

qf_dataset.tail(), qf_dataset.shape


(            quantile_lower       q25       q50       q75  quantile_upper  \
 Date                                                                       
 2025-04-24       -0.006618 -0.003052 -0.000537  0.002935        0.006888   
 2025-04-25       -0.006618 -0.001962  0.000188  0.002762        0.006262   
 2025-04-28       -0.006508 -0.003019 -0.000137  0.002562        0.006149   
 2025-04-29       -0.006618 -0.001958  0.000218  0.002706        0.006149   
 2025-04-30       -0.006618 -0.002011  0.000117  0.003095        0.006149   
 
             quantile_width  quantile_skew  asymmetry  
 Date                                                  
 2025-04-24        0.013507       0.000957   0.450225  
 2025-04-25        0.012880       0.000425   0.528412  
 2025-04-28        0.012657      -0.000181   0.503346  
 2025-04-29        0.012767       0.000313   0.535454  
 2025-04-30        0.012767       0.000849   0.527569  ,
 (205, 8))

## 6) Padronização via função única para indexação

In [6]:
def standardize_dataset(df):
    df = df.sort_index()

    df.index = pd.to_datetime(df.index)
    df.index = df.index.astype("datetime64[us]")
    df.index.name = "timestamp"

    df = df.astype(np.float32)

    assert isinstance(df.index, pd.DatetimeIndex)
    assert df.index.dtype == "datetime64[us]"
    assert df.index.name == "timestamp"
    assert df.index.is_monotonic_increasing
    assert not df.index.has_duplicates

    return df

In [7]:
qf_dataset = standardize_dataset(qf_dataset)

In [8]:
qf_dataset

,quantile_lower,q25,q50,q75,quantile_upper,quantile_width,quantile_skew,asymmetry
timestamp,,,,,,,,
2024-07-18,-0.006797,-0.002225,-0.000400,0.002021,0.006149,0.012946,0.000596,0.494136
2024-07-19,-0.006797,-0.002202,-0.000079,0.001899,0.006149,0.012946,-0.000144,0.518938
2024-07-22,-0.006797,-0.002277,-0.000125,0.002120,0.006149,0.012946,0.000094,0.515370
2024-07-23,-0.006758,-0.002260,-0.000174,0.002118,0.006149,0.012906,0.000207,0.510073
2024-07-24,-0.006758,-0.002685,-0.000176,0.002007,0.006149,0.012906,-0.000327,0.509983
...,...,...,...,...,...,...,...,...
2025-04-24,-0.006618,-0.003052,-0.000537,0.002935,0.006888,0.013507,0.000957,0.450225
2025-04-25,-0.006618,-0.001962,0.000188,0.002762,0.006262,0.012880,0.000425,0.528412
2025-04-28,-0.006508,-0.003019,-0.000137,0.002562,0.006149,0.012657,-0.000181,0.503346


## 7) Salvamento (opcional)


In [9]:
OUTPUT_PATH = "C:/projects/Libellula/data/processed/qf/qf_features.parquet"
qf_dataset.to_parquet(OUTPUT_PATH, index=True)
OUTPUT_PATH

actual_returns = pd.Series(splits["y_val"], index=qf_dataset.index)
quantile_columns = {0.05: "quantile_lower", 0.25: "q25", 0.50: "q50", 0.75: "q75", 0.95: "quantile_upper"}
calibration_rows = []
for quantile, column in quantile_columns.items():
    forecast = qf_dataset[column]
    error = actual_returns - forecast
    pinball_loss = np.maximum(quantile * error, (quantile - 1) * error).mean()
    calibration_rows.append({"quantile": quantile, "empirical_coverage": (actual_returns <= forecast).mean(), "pinball_loss": pinball_loss})

quantile_diagnostics = pd.DataFrame(calibration_rows).set_index("quantile")
print(quantile_diagnostics)


'C:/projects/Libellula/data/processed/qf/qf_features.parquet'